# 🚀 Búsqueda Informada Heurística: Greedy Best-First y A*

## Objetivo de aprendizaje
Comprender cómo una **heurística** permite orientar una búsqueda hacia un objetivo y comparar dos algoritmos de búsqueda informada: **Greedy Best-First Search** y **A\***.

Al finalizar podremos identificar la diferencia entre **h(n)**, **g(n)** y **f(n)**, ejecutar ambos algoritmos y comparar la ruta encontrada y su costo.

## 1. ¿Qué es una búsqueda informada o heurística?

A diferencia de BFS o DFS, una búsqueda informada utiliza conocimiento adicional del problema para decidir qué nodo explorar primero. Ese conocimiento se representa mediante una **función heurística**.

- **g(n):** costo real acumulado desde el nodo inicial hasta el nodo `n`.
- **h(n):** estimación del costo que falta desde `n` hasta el objetivo.
- **f(n) = g(n) + h(n):** estimación del costo total de una solución que pasa por `n`.

**Greedy Best-First** prioriza únicamente `h(n)`. **A\*** combina `g(n)` y `h(n)`.

## 2. Briefing de la misión 🛰️

Nuestra misión consiste en encontrar una ruta eficiente en combustible desde la **Tierra** hasta el exoplaneta **Kepler-186f**.

Cada conexión del mapa tiene un costo real de combustible. Además, cada punto posee una heurística que estima cuánto falta para llegar al destino.

In [9]:
# Grafo ponderado: destino -> costo de combustible
rutas_espaciales = {
    'Tierra': {'Marte': 4, 'Estacion Alfa': 3},
    'Marte': {'Jupiter': 5},
    'Estacion Alfa': {'Jupiter': 2, 'Cinturon Asteroides': 6},
    'Jupiter': {'Saturno': 3},
    'Cinturon Asteroides': {'Kepler-186f': 7},
    'Saturno': {'Kepler-186f': 4},
    'Kepler-186f': {}
}

rutas_espaciales

{'Tierra': {'Marte': 4, 'Estacion Alfa': 3},
 'Marte': {'Jupiter': 5},
 'Estacion Alfa': {'Jupiter': 2, 'Cinturon Asteroides': 6},
 'Jupiter': {'Saturno': 3},
 'Cinturon Asteroides': {'Kepler-186f': 7},
 'Saturno': {'Kepler-186f': 4},
 'Kepler-186f': {}}

## 3. Función heurística `h(n)`

La heurística funciona como una **pista**: estima el costo restante hasta Kepler-186f. Cuanto menor sea `h(n)`, aparentemente más cerca estamos del objetivo.

> Importante: una heurística es una estimación; no representa necesariamente el costo real del viaje.

In [14]:
heuristica = {
    'Tierra': 10,
    'Marte': 5,
    'Estacion Alfa': 8,
    'Jupiter': 9,
    'Cinturon Asteroides': 3,
    'Saturno': 6,
    'Kepler-186f': 0
}

heuristica

{'Tierra': 10,
 'Marte': 5,
 'Estacion Alfa': 8,
 'Jupiter': 9,
 'Cinturon Asteroides': 3,
 'Saturno': 6,
 'Kepler-186f': 0}

## 4. Greedy Best-First Search: el piloto ambicioso

Greedy selecciona el nodo que **parece estar más cerca del objetivo**.

### `prioridad(n) = h(n)`

Esto puede hacerlo rápido, pero también puede llevarlo por una ruta costosa porque **ignora cuánto combustible ya se ha gastado**.

In [15]:
import heapq

def greedy_best_first(graph, start, goal, heuristics, mostrar_pasos=False):
    priority_queue = [(heuristics[start], [start])]
    visited = set()
    paso = 1

    while priority_queue:
        h, path = heapq.heappop(priority_queue)
        node = path[-1]

        if node in visited:
            continue
        visited.add(node)

        if mostrar_pasos:
            print(f"Paso {paso}: explorar {node:20s} | h={h:2d} | camino={path}")
            paso += 1

        if node == goal:
            return path

        for neighbor in graph[node]:
            if neighbor not in visited:
                heapq.heappush(
                    priority_queue,
                    (heuristics[neighbor], path + [neighbor])
                )

    return None

### Ejecutamos Greedy

Desde la Tierra prefiere **Marte**, porque `h(Marte)=8` es menor que `h(Estacion Alfa)=9`.

In [16]:
ruta_greedy = greedy_best_first(
    rutas_espaciales, 'Tierra', 'Kepler-186f',
    heuristica, mostrar_pasos=True
)

print("\nRuta Greedy:", ruta_greedy)

Paso 1: explorar Tierra               | h=10 | camino=['Tierra']
Paso 2: explorar Marte                | h= 5 | camino=['Tierra', 'Marte']
Paso 3: explorar Estacion Alfa        | h= 8 | camino=['Tierra', 'Estacion Alfa']
Paso 4: explorar Cinturon Asteroides  | h= 3 | camino=['Tierra', 'Estacion Alfa', 'Cinturon Asteroides']
Paso 5: explorar Kepler-186f          | h= 0 | camino=['Tierra', 'Estacion Alfa', 'Cinturon Asteroides', 'Kepler-186f']

Ruta Greedy: ['Tierra', 'Estacion Alfa', 'Cinturon Asteroides', 'Kepler-186f']


## 5. Costo real de la ruta

Greedy no usa los costos para decidir, pero podemos calcularlos al final para evaluar la calidad de la solución.

In [17]:
def costo_ruta(graph, path):
    return sum(
        graph[path[i]][path[i + 1]]
        for i in range(len(path) - 1)
    )

costo_greedy = costo_ruta(rutas_espaciales, ruta_greedy)
print("Costo total Greedy:", costo_greedy, "unidades de combustible")

Costo total Greedy: 16 unidades de combustible


### Resultado de Greedy

**Ruta esperada:**

`Tierra → Marte → Jupiter → Saturno → Kepler-186f`

**Costo:** `4 + 5 + 3 + 4 = 16`

La elección inicial parecía prometedora por su heurística, pero terminó siendo más costosa.

## 6. A*: el planificador de rutas óptimo

A\* considera simultáneamente lo que ya hemos gastado y lo que estimamos que falta:

### `f(n) = g(n) + h(n)`

Por ello no se deja llevar únicamente por el nodo que *parece* más cercano.

In [18]:
def a_star_search(graph, start, goal, heuristics, mostrar_pasos=False):
    # Cada elemento es: (f, g, camino)
    priority_queue = [(heuristics[start], 0, [start])]
    mejor_g = {start: 0}
    paso = 1

    while priority_queue:
        f, g, path = heapq.heappop(priority_queue)
        node = path[-1]

        # Si existe una entrada más reciente y barata, ignoramos esta.
        if g != mejor_g.get(node):
            continue

        if mostrar_pasos:
            print(
                f"Paso {paso}: explorar {node:20s} | "
                f"g={g:2d} | h={heuristics[node]:2d} | "
                f"f={f:2d} | camino={path}"
            )
            paso += 1

        if node == goal:
            return path, g

        for neighbor, cost in graph[node].items():
            new_g = g + cost

            if new_g < mejor_g.get(neighbor, float('inf')):
                mejor_g[neighbor] = new_g
                new_f = new_g + heuristics[neighbor]

                heapq.heappush(
                    priority_queue,
                    (new_f, new_g, path + [neighbor])
                )

    return None, -1

### Ejecutamos A*

A\* compara `f(n)`. Así equilibra el costo acumulado `g(n)` con la estimación restante `h(n)`.

In [19]:
ruta_astar, costo_astar = a_star_search(
    rutas_espaciales, 'Tierra', 'Kepler-186f',
    heuristica, mostrar_pasos=True
)

print("\nRuta A*:", ruta_astar)
print("Costo total A*:", costo_astar, "unidades de combustible")

Paso 1: explorar Tierra               | g= 0 | h=10 | f=10 | camino=['Tierra']
Paso 2: explorar Marte                | g= 4 | h= 5 | f= 9 | camino=['Tierra', 'Marte']
Paso 3: explorar Estacion Alfa        | g= 3 | h= 8 | f=11 | camino=['Tierra', 'Estacion Alfa']
Paso 4: explorar Cinturon Asteroides  | g= 9 | h= 3 | f=12 | camino=['Tierra', 'Estacion Alfa', 'Cinturon Asteroides']
Paso 5: explorar Jupiter              | g= 5 | h= 9 | f=14 | camino=['Tierra', 'Estacion Alfa', 'Jupiter']
Paso 6: explorar Saturno              | g= 8 | h= 6 | f=14 | camino=['Tierra', 'Estacion Alfa', 'Jupiter', 'Saturno']
Paso 7: explorar Kepler-186f          | g=12 | h= 0 | f=12 | camino=['Tierra', 'Estacion Alfa', 'Jupiter', 'Saturno', 'Kepler-186f']

Ruta A*: ['Tierra', 'Estacion Alfa', 'Jupiter', 'Saturno', 'Kepler-186f']
Costo total A*: 12 unidades de combustible


### Resultado de A*

**Ruta esperada:**

`Tierra → Estacion Alfa → Jupiter → Saturno → Kepler-186f`

**Costo:** `3 + 2 + 3 + 4 = 12`

A\* encuentra una alternativa de menor costo porque combina la información real `g(n)` con la estimación `h(n)`.

## 7. Comparación final

In [20]:
print("COMPARACIÓN DE ALGORITMOS")
print("-" * 70)

print("Greedy | Ruta:", " -> ".join(ruta_greedy))
print("       | Costo:", costo_greedy)
print()

print("A*     | Ruta:", " -> ".join(ruta_astar))
print("       | Costo:", costo_astar)
print()

print("Ahorro de A*:", costo_greedy - costo_astar,
      "unidades de combustible")

COMPARACIÓN DE ALGORITMOS
----------------------------------------------------------------------
Greedy | Ruta: Tierra -> Estacion Alfa -> Cinturon Asteroides -> Kepler-186f
       | Costo: 16

A*     | Ruta: Tierra -> Estacion Alfa -> Jupiter -> Saturno -> Kepler-186f
       | Costo: 12

Ahorro de A*: 4 unidades de combustible


| Algoritmo | Prioridad | ¿Considera costo recorrido? | Costo |
|---|---|---:|---:|
| Greedy Best-First | `h(n)` | No | 16 |
| A* | `g(n) + h(n)` | Sí | 12 |

## 8. Debriefing de la misión

Greedy tomó una decisión local: **Marte parecía más cercano** según la heurística. A\*, en cambio, evaluó el equilibrio entre el combustible gastado y el costo estimado restante.

En este problema, A\* consigue una ruta que consume **4 unidades menos**, un ahorro del **25 % respecto al costo de la ruta Greedy**.

### ⚠️ Precisión importante sobre A*

A\* no es óptimo con cualquier heurística. La garantía depende de las propiedades de `h(n)` y de la implementación. Una heurística **admisible** nunca sobreestima el costo real restante. En búsqueda sobre grafos también resulta especialmente útil que sea **consistente**.

La idea central del ejercicio es observar cómo `g(n) + h(n)` puede producir decisiones mejor fundamentadas que utilizar solamente `h(n)`.

## 9. Actividad para el estudiante 🧠

1. Cambia `h('Marte')` de `8` a `4`. ¿Qué ocurre con Greedy?
2. Cambia el costo `Tierra → Estacion Alfa` de `3` a `8`. ¿A\* mantiene la misma ruta?
3. Agrega una ruta directa `Marte → Kepler-186f` con costo `15`.
4. Ejecuta nuevamente ambos algoritmos y compara los resultados.
5. Explica con tus propias palabras por qué una heurística útil puede reducir la cantidad de alternativas que deben explorarse.

## 10. Preguntas de cierre

- ¿Cuál es la diferencia entre `g(n)` y `h(n)`?
- ¿Por qué Greedy puede encontrar una solución que no sea la más barata?
- ¿Qué representa `f(n)` en A\*?
- ¿Qué significa que una heurística sea admisible?
- ¿En qué aplicaciones reales utilizarías una búsqueda heurística?

### Idea clave
Una búsqueda informada **no explora a ciegas**: utiliza información del problema para decidir qué alternativas parecen más prometedoras.